# Отчет NL2SQL: Synthetic E-commerce Benchmark

Аналитический notebook по результатам локального synthetic benchmark-сценария `synthetic_ecommerce`. Ниже рассматриваются агрегированные результаты режимов `ea` и `pass_k`, распределение качества по уровням сложности и поведение моделей на уровне отдельных запросов.


## 1. Подготовка и источники данных

В данном разделе задаются пути к synthetic-результатам, настраивается окружение для анализа и загружаются агрегированные и детализированные артефакты прогонов.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent):
    if (candidate / 'nl2sql').exists() and (candidate / 'shared').exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RESULTS_ROOT = PROJECT_ROOT / 'results' / 'nl2sql' / 'synthetic_benchmark'
EA_ROOT = RESULTS_ROOT
PASSK_ROOT = RESULTS_ROOT / 'pass_k'
FIGURES_DIR = RESULTS_ROOT / 'figures' / 'synthetic_analysis'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_META = {
    'm1_deepseek': {'display_name': 'DeepSeek', 'model_group': 'M1'},
    'm1_chatgpt': {'display_name': 'ChatGPT', 'model_group': 'M1'},
    'm2_defog': {'display_name': 'Defog-Llama3-SQLCoder-8B', 'model_group': 'M2'},
    'm2_hrida': {'display_name': 'Hrida-T2SQL', 'model_group': 'M2'},
    'm2_arctic': {'display_name': 'Arctic-Text2SQL-R1-7B', 'model_group': 'M2'},
}

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

print(f'Project root: {PROJECT_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Figures dir: {FIGURES_DIR}')


In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def iter_result_dirs(root: Path, mode: str):
    if mode == 'ea':
        dirs = [p for p in root.iterdir() if p.is_dir() and p.name not in {'pass_k', 'smoke', 'figures'}]
    else:
        dirs = [p for p in root.iterdir() if p.is_dir()]
    return sorted(dirs)


def build_summary_rows(result_dirs, mode: str):
    overall_rows = []
    difficulty_rows = []
    detailed_rows = []

    for result_dir in result_dirs:
        model_key = result_dir.name
        meta = MODEL_META.get(model_key, {'display_name': model_key, 'model_group': 'other'})
        results_path = result_dir / 'results.json'
        details_path = result_dir / 'detailed_results.json'
        if not results_path.exists() or not details_path.exists():
            continue

        summary = load_json(results_path)
        detailed = load_json(details_path)

        overall_row = {
            'model_key': model_key,
            'model_display_name': meta['display_name'],
            'model_group': meta['model_group'],
            'mode': mode,
            **summary['overall'],
            'n_samples': len(detailed),
        }
        overall_rows.append(overall_row)

        for difficulty, metrics in summary.get('by_difficulty', {}).items():
            difficulty_rows.append({
                'model_key': model_key,
                'model_display_name': meta['display_name'],
                'model_group': meta['model_group'],
                'mode': mode,
                'difficulty': difficulty,
                **metrics,
            })

        for sample_index, row in enumerate(detailed, start=1):
            candidate_results = row.get('candidate_results') or []
            detailed_rows.append({
                'model_key': model_key,
                'model_display_name': meta['display_name'],
                'model_group': meta['model_group'],
                'mode': mode,
                'sample_index': sample_index,
                'question': row.get('question'),
                'sql_gt': row.get('sql_gt'),
                'sql_pred': row.get('sql_pred'),
                'difficulty': row.get('difficulty'),
                'valid': bool(row.get('valid')),
                'execution_match': bool(row.get('execution_match')),
                'inference_error': row.get('inference_error'),
                'prediction_error': row.get('prediction_error'),
                'ground_truth_error': row.get('ground_truth_error'),
                'n_candidates': len(row.get('sql_candidates') or []),
                'n_valid_candidates': sum(1 for c in candidate_results if c.get('valid')),
                'n_matching_candidates': sum(1 for c in candidate_results if c.get('execution_match')),
                'any_candidate_match': any(c.get('execution_match') for c in candidate_results) if candidate_results else bool(row.get('execution_match')),
            })

    overall_df = pd.DataFrame(overall_rows)
    difficulty_df = pd.DataFrame(difficulty_rows)
    detailed_df = pd.DataFrame(detailed_rows)
    return overall_df, difficulty_df, detailed_df


ea_overall_df, ea_difficulty_df, ea_detail_df = build_summary_rows(iter_result_dirs(EA_ROOT, 'ea'), mode='ea')
passk_overall_df, passk_difficulty_df, passk_detail_df = build_summary_rows(iter_result_dirs(PASSK_ROOT, 'pass_k'), mode='pass_k')

pass_metric_cols = sorted(
    [col for col in passk_overall_df.columns if col.startswith('pass@') and '_ci_' not in col and '_q' not in col],
    key=lambda col: int(col.split('@', 1)[1]),
)

print('EA overall:', ea_overall_df.shape)
print('EA difficulty:', ea_difficulty_df.shape)
print('EA detail:', ea_detail_df.shape)
print('Pass@K overall:', passk_overall_df.shape)
print('Pass@K difficulty:', passk_difficulty_df.shape)
print('Pass@K detail:', passk_detail_df.shape)

display(ea_overall_df.sort_values(['model_group', 'model_display_name']).round(4))
display(passk_overall_df[['model_display_name', 'model_group', 'execution_accuracy', 'valid_sql_rate', *pass_metric_cols, 'n_samples']].sort_values(['model_group', 'model_display_name']).round(4))


## 2. Обзор доступных прогонов

Synthetic benchmark выполняется на фиксированном наборе запросов, поэтому на уровне обзора важно проверить полноту доступных прогонов, объем выборки и базовые характеристики самих benchmark-примеров.


In [ ]:
dataset_df = (
    ea_detail_df[['sample_index', 'question', 'sql_gt', 'difficulty']]
    .sort_values('sample_index')
    .drop_duplicates(subset=['sample_index'])
    .copy()
)
dataset_df['question_len'] = dataset_df['question'].str.len()
dataset_df['gold_sql_len'] = dataset_df['sql_gt'].str.len()

run_inventory_df = pd.concat([
    ea_overall_df[['model_display_name', 'model_group', 'mode', 'n_samples']],
    passk_overall_df[['model_display_name', 'model_group', 'mode', 'n_samples']],
], ignore_index=True).sort_values(['mode', 'model_group', 'model_display_name'])

dataset_overview_df = pd.DataFrame([
    {
        'n_samples': len(dataset_df),
        'mean_question_len': dataset_df['question_len'].mean(),
        'median_question_len': dataset_df['question_len'].median(),
        'mean_gold_sql_len': dataset_df['gold_sql_len'].mean(),
        'median_gold_sql_len': dataset_df['gold_sql_len'].median(),
    }
])

display(run_inventory_df)
display(dataset_df['difficulty'].value_counts().rename_axis('difficulty').reset_index(name='count'))
display(dataset_overview_df.round(2))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.countplot(data=dataset_df, x='difficulty', order=['easy', 'medium', 'hard'], ax=axes[0], palette='Set2')
axes[0].set_title('Распределение запросов по сложности')
axes[0].set_xlabel('Difficulty')
axes[0].set_ylabel('Count')

sns.histplot(data=dataset_df, x='question_len', bins=15, ax=axes[1], color='#4c72b0')
axes[1].set_title('Длина NL-вопросов')
axes[1].set_xlabel('Question length')

sns.histplot(data=dataset_df, x='gold_sql_len', bins=15, ax=axes[2], color='#55a868')
axes[2].set_title('Длина эталонного SQL')
axes[2].set_xlabel('Gold SQL length')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'synthetic_report_01_dataset_overview.png', dpi=300, bbox_inches='tight')
plt.show()


## 3. Основные результаты режима EA

В данном разделе рассматриваются single-shot результаты `ea`, то есть качество первого сгенерированного SQL-кандидата без многократных попыток генерации.


In [ ]:
ea_main_df = ea_overall_df[['model_display_name', 'model_group', 'execution_accuracy', 'valid_sql_rate', 'n_samples']].sort_values(['model_group', 'execution_accuracy'], ascending=[True, False])
display(ea_main_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=ea_overall_df, x='model_display_name', y='execution_accuracy', hue='model_group', dodge=False, ax=axes[0])
axes[0].set_title('Execution Accuracy in EA mode')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('EA')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)

sns.barplot(data=ea_overall_df, x='model_display_name', y='valid_sql_rate', hue='model_group', dodge=False, ax=axes[1])
axes[1].set_title('Valid SQL Rate in EA mode')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Valid SQL rate')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=25)

for ax in axes:
    if ax.legend_ is not None:
        ax.legend_.remove()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(labels))
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(FIGURES_DIR / 'synthetic_report_02_ea_overall.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Разложение EA по уровням сложности

Поскольку benchmark содержит сбалансированное распределение `easy`, `medium` и `hard` запросов, отдельный анализ по сложности позволяет увидеть, в какой части набора происходит основное снижение качества.


In [ ]:
ea_difficulty_pivot = ea_difficulty_df.pivot(index='model_display_name', columns='difficulty', values='execution_accuracy')[['easy', 'medium', 'hard']]
display(ea_difficulty_pivot.round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=ea_difficulty_df,
    x='difficulty',
    y='execution_accuracy',
    hue='model_display_name',
    order=['easy', 'medium', 'hard'],
    ax=axes[0],
)
axes[0].set_title('EA by difficulty')
axes[0].set_xlabel('Difficulty')
axes[0].set_ylabel('Execution accuracy')
axes[0].set_ylim(0, 1.05)

sns.heatmap(ea_difficulty_pivot, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('EA heatmap by model and difficulty')
axes[1].set_xlabel('Difficulty')
axes[1].set_ylabel('Model')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'synthetic_report_03_ea_by_difficulty.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. Основные результаты режима Pass@K

В режиме `pass_k` для каждого примера генерируется несколько кандидатов. Это позволяет оценить, насколько качество повышается при переходе от одного ответа к множеству независимых попыток.


In [ ]:
passk_main_cols = ['model_display_name', 'model_group', 'execution_accuracy', 'valid_sql_rate', *pass_metric_cols, 'n_samples']
display(passk_overall_df[passk_main_cols].sort_values(['model_group', 'model_display_name']).round(4))

passk_plot_df = passk_overall_df.melt(
    id_vars=['model_display_name', 'model_group'],
    value_vars=['execution_accuracy', *pass_metric_cols],
    var_name='metric',
    value_name='score',
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=passk_overall_df, x='model_display_name', y='valid_sql_rate', hue='model_group', dodge=False, ax=axes[0])
axes[0].set_title('Valid SQL Rate in Pass@K mode')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Valid SQL rate')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)

sns.barplot(data=passk_plot_df, x='metric', y='score', hue='model_display_name', ax=axes[1])
axes[1].set_title('EA and Pass@K by model')
axes[1].set_xlabel('Metric')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1.05)

if axes[0].legend_ is not None:
    axes[0].legend_.remove()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'synthetic_report_04_passk_overall.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Прирост от многократной генерации

Ниже сопоставляются результаты `ea` и `pass_k` для одних и тех же моделей. Такой анализ показывает, насколько наличие нескольких кандидатов компенсирует ошибки первого ответа и где эффект от многократной генерации выражен сильнее.


In [ ]:
ea_vs_passk_df = ea_overall_df[['model_key', 'model_display_name', 'model_group', 'execution_accuracy']].merge(
    passk_overall_df[['model_key', *pass_metric_cols]],
    on='model_key',
    how='inner',
)
for metric in pass_metric_cols:
    ea_vs_passk_df[f'{metric}_gain_vs_ea'] = ea_vs_passk_df[metric] - ea_vs_passk_df['execution_accuracy']

display(ea_vs_passk_df.sort_values(['model_group', 'model_display_name']).round(4))

if pass_metric_cols:
    final_metric = pass_metric_cols[-1]
    plt.figure(figsize=(10, 6))
    gain_df = ea_vs_passk_df[['model_display_name', f'{final_metric}_gain_vs_ea']].sort_values(f'{final_metric}_gain_vs_ea', ascending=False)
    sns.barplot(data=gain_df, x='model_display_name', y=f'{final_metric}_gain_vs_ea', color='#c44e52')
    plt.title(f'Gain of {final_metric} over EA')
    plt.xlabel('Model')
    plt.ylabel('Absolute gain')
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'synthetic_report_05_passk_gain.png', dpi=300, bbox_inches='tight')
    plt.show()

passk_difficulty_metric = pass_metric_cols[-1] if pass_metric_cols else 'execution_accuracy'
passk_difficulty_pivot = passk_difficulty_df.pivot(index='model_display_name', columns='difficulty', values=passk_difficulty_metric)[['easy', 'medium', 'hard']]
display(passk_difficulty_pivot.round(4))

plt.figure(figsize=(10, 6))
sns.heatmap(passk_difficulty_pivot, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1)
plt.title(f'{passk_difficulty_metric} by model and difficulty')
plt.xlabel('Difficulty')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'synthetic_report_06_passk_by_difficulty.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Анализ ошибок на уровне запросов

Завершающий раздел фиксирует распределение ошибок по моделям и выделяет запросы, которые вызывают наибольшие затруднения. Для `pass_k` дополнительно оценивается, в скольких случаях наличие нескольких кандидатов позволило восстановить корректный результат.


In [ ]:
ea_error_summary_df = ea_detail_df.groupby(['model_display_name', 'difficulty']).agg(
    total=('sample_index', 'count'),
    valid_sql=('valid', 'sum'),
    execution_match=('execution_match', 'sum'),
).reset_index()
ea_error_summary_df['invalid_sql'] = ea_error_summary_df['total'] - ea_error_summary_df['valid_sql']
ea_error_summary_df['misses'] = ea_error_summary_df['total'] - ea_error_summary_df['execution_match']
display(ea_error_summary_df.sort_values(['model_display_name', 'difficulty']))

hardest_queries_df = (
    ea_detail_df.groupby(['sample_index', 'difficulty', 'question'])
    .agg(models_solved=('execution_match', 'sum'), n_models=('model_key', 'nunique'))
    .reset_index()
)
hardest_queries_df['solve_rate'] = hardest_queries_df['models_solved'] / hardest_queries_df['n_models']
display(hardest_queries_df.sort_values(['solve_rate', 'difficulty', 'sample_index']).head(10))

recovery_df = passk_detail_df[['model_key', 'model_display_name', 'sample_index', 'difficulty', 'execution_match', 'any_candidate_match']].merge(
    ea_detail_df[['model_key', 'sample_index', 'execution_match']].rename(columns={'execution_match': 'ea_execution_match'}),
    on=['model_key', 'sample_index'],
    how='left',
)
recovery_df['recovered_in_passk'] = (~recovery_df['ea_execution_match']) & (recovery_df['any_candidate_match'])

recovery_summary_df = recovery_df.groupby('model_display_name').agg(
    recovered_cases=('recovered_in_passk', 'sum'),
    total_samples=('sample_index', 'count'),
    passk_any_hit=('any_candidate_match', 'mean'),
).reset_index().sort_values('recovered_cases', ascending=False)
display(recovery_summary_df.round(4))

recovered_examples_df = recovery_df[recovery_df['recovered_in_passk']].merge(
    passk_detail_df[['model_key', 'sample_index', 'question', 'sql_gt', 'sql_pred', 'n_matching_candidates']],
    on=['model_key', 'sample_index'],
    how='left',
)
display(recovered_examples_df[['model_display_name', 'difficulty', 'sample_index', 'n_matching_candidates', 'question']].head(15))
